In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from utils.threebody_solver import PlanetMassModel_WithKeplerModel

from utils import TensorCode_util as tc
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from utils import initvalues_util
import copy
import keras
import joblib

%load_ext autoreload
%autoreload 2

In [ ]:
model_name = 'DT'
num_bodies = 2
equal_masses = False

In [ ]:
trained_model_sub_name = ''

if equal_masses:
    trained_model_sub_name += '_equal_masses'

In [ ]:
if model_name in ['RF', 'DT']:
    trained_model = joblib.load(f'trained_models/{model_name}{trained_model_sub_name}_trained.joblib')
else:
    trained_model = keras.models.load_model(f'trained_models/{model_name}{trained_model_sub_name}_trained.keras')


In [ ]:
model = [model_name, trained_model]
model = []
steps = 5

In [ ]:
tau, n, m, r, v = initvalues_util.initValues(num_bodies, equal_mass=equal_masses)

r_t = [r]
v_t = [v]

for k in range(steps):
    rv = np.column_stack([r, v])
    
    rv1 = tc.do_step_wrapper_tfmodel(tau, n, m, rv, model = [])
    r1, v1 = tf.split(rv1, num_or_size_splits=2, axis=-1)
    r1, v1 = r1.numpy(), v1.numpy()

    r = copy.deepcopy(r1)
    v = copy.deepcopy(v1)

r_t.append(r1)
v_t.append(v1)

r_t = np.stack(r_t)
v_t = np.stack(v_t)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for i_planet in range(n):
    ax.plot(*r_t[:, i_planet, :2].T, '.-', markevery=10)

In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection='3d')
for i_planet in range(n):
    ax.scatter(r_t[:, i_planet, 0], r_t[:, i_planet, 1], r_t[:, i_planet, 2])

# ax.scatter(center_of_mass[:,0], center_of_mass[:,1], center_of_mass[:,2])

In [ ]:
n_epochs = 2000
batch_size = 1
learning_rate = 50.0
num_of_steps = steps

m_true = m.copy()
m_initial = np.array(num_bodies*[1])
m_initial[0] = 1.0
m_initial[1] = 2.0

mask = np.array(num_bodies*[1])
mask[0] = 0

print("m_true", m_true)
print("m_initial", m_initial)

pmm = PlanetMassModel_WithKeplerModel(time_step=tau, initial_masses=m_initial, num_of_steps=num_of_steps, mass_penalty=0, trained_model=model, mask = mask)

optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
pmm.compile(loss=pmm.loss_fn, optimizer=optimizer)

In [ ]:
rv_t = np.dstack([r_t, v_t])
dataset = np.dstack((rv_t[:-1, ...], rv_t[1:, ...]))

In [ ]:
tf.config.run_functions_eagerly(True)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor='loss', patience=50, min_delta=0, restore_best_weights=True)

# record_history = RecordHistory(model=pmm)
# print_masses = PrintMasses(model = pmm)

historian = pmm.fit(dataset, epochs=n_epochs, callbacks=[early_stopping], batch_size=batch_size)

In [ ]:
print("initial masses", m_initial, f"error {np.linalg.norm(m_initial - m_true):3E}")
print("final masses", pmm.masses, f"error {np.linalg.norm(pmm.masses - m_true):3E}")
print("true masses", m_true)

print("relative errors inital", [f"{(np.abs(p[0]-p[1])/p[1]):3E}" for p in zip(m_initial, m_true)])
print("relative errors final", [f"{(np.abs(p[0]-p[1])/p[1]):3E}" for p in zip(pmm.masses, m_true)])

fig, ax = plt.subplots(1, 1, figsize=(5, 3))
ax.semilogy(historian.history['loss'])
plt.xlabel("Epochs")
plt.ylabel("SSE")